# Ajuste de la Curva de Transmisión de un Filtro Óptico

**Cuatrimestre:** Primer cuatrimestre de 2024

---

Un experimento de óptica consiste en medir la transmisión de un filtro pasa-altos en función de la longitud de onda $\lambda$. Para cada longitud de onda, se lanzan $n = 50$ fotones contra el filtro y se cuenta cuántos lo atraviesan ($k$). La eficiencia de transmisión se modela como una función sigmoide:

$$
\varepsilon(\lambda; a, b) = \frac{1}{1 + \exp\!\left(-\dfrac{\lambda - a}{b}\right)}
$$

donde $a$ es la longitud de onda de corte (donde $\varepsilon = 0.5$) y $b$ controla la pendiente de la transición. La tabla siguiente resume los datos del experimento.

In [3]:
%pip install -q likefit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy as sp
import scipy.stats as st
import likefit

ModuleNotFoundError: No module named 'likefit'

In [ ]:
# Datos experimentales
wavelength = np.arange(400, 810, 20, dtype=float)   # longitud de onda (nm)
ntrials    = 50                                       # fotones por punto
total      = np.full(wavelength.shape, ntrials)
observed   = np.array([0, 0, 1, 0, 1, 2, 1, 3, 4, 5,
                        9, 13, 18, 24, 28, 31, 37, 42, 46, 48, 50])

data = pd.DataFrame({'Wavelength (nm)': wavelength,
                     'Total':           total,
                     'Transmitted':     observed})
data

**1. Estimador de la transmisión para cada longitud de onda**

Con la longitud de onda fija, la fracción de fotones transmitidos es el estimador natural de la eficiencia:
$$
\hat{\varepsilon} = \frac{k}{n}
$$

In [ ]:
ef = data['Transmitted'] / data['Total']
ef

**2. Gráfico del estimador en función de la longitud de onda**

In [ ]:
fig, ax = plt.subplots()
ax.plot(data['Wavelength (nm)'], ef, 'o')
ax.set_xlabel('Longitud de onda $\\lambda$ (nm)')
ax.set_ylabel('Transmisión $\\hat{\\varepsilon}$')
ax.set_title('Transmisión del filtro vs longitud de onda')
ax.grid(True)
plt.tight_layout()
plt.show()

**3. Modelo sigmoide**

$$
\varepsilon(\lambda; a, b) = \frac{1}{1 + e^{-(\lambda - a)/b}}
$$

- $a$: longitud de onda de corte (nm) — donde la transmisión es del 50%
- $b$: parámetro de pendiente (nm) — cuanto más chico, más abrupta la transición

In [ ]:
def trans_model(lam, param):
    """Modelo sigmoide para likefit (param es array [a, b])."""
    return 1 / (1 + np.exp(-(lam - param[0]) / param[1]))

**4. Ajuste con función de costo Binomial**

Primero obtenemos parámetros iniciales con `scipy.optimize.curve_fit`, y luego hacemos el ajuste por máxima verosimilitud binomial con `likefit`.

In [ ]:
# Ajuste previo con scipy para obtener buenos parámetros iniciales
def trans_model_2(lam, a, b):
    return 1 / (1 + np.exp(-(lam - a) / b))

init_param, _ = sp.optimize.curve_fit(
    f=trans_model_2,
    xdata=data['Wavelength (nm)'],
    ydata=ef,
    p0=[600, 30]
)

print(f'a inicial = {init_param[0]:.3f} nm')
print(f'b inicial = {init_param[1]:.3f} nm')

In [ ]:
# Ajuste por máxima verosimilitud binomial con likefit
fitter = likefit.Binomial(
    data['Wavelength (nm)'],
    data['Total'],
    data['Transmitted'],
    trans_model
)

fit_status = fitter.fit(init_param)
fitter.print_results()

**5. Parámetros estimados y errores**

In [ ]:
estimators = fitter.get_estimators()
errors     = fitter.get_errors()

print(f'Longitud de onda de corte:  a = {estimators[0]:.2f} ± {errors[0]:.2f} nm')
print(f'Parámetro de pendiente:     b = {estimators[1]:.2f} ± {errors[1]:.2f} nm')

**6. Correlación entre los parámetros $a$ y $b$**

In [ ]:
cov = fitter.get_covariance_matrix()
correlacion = cov[0, 1]

print(f'Matriz de covarianza:')
print(cov)
print(f'\nCorrelación entre a y b = {correlacion:.4f}')

**7. Región de confianza $1\sigma$ de los parámetros**

In [ ]:
fitter.plot_confidence_regions(parx_index=0, pary_index=1, nsigma=1)
plt.xlabel('$a$ (nm)')
plt.ylabel('$b$ (nm)')
plt.title('Región de confianza $1\\sigma$')
plt.show()

**8. Bondad del ajuste: $\chi^2$ y p-valor**

In [ ]:
chi2   = fitter.get_chi_square()
pvalor = fitter.get_pvalue()

print(f'χ² del ajuste = {chi2:.3f}')
print(f'p-valor       = {pvalor:.4f}')

In [ ]:
# Valor crítico de chi2 para df = N_puntos - 2 parámetros
df = len(data['Wavelength (nm)']) - 2
alpha = 0.05
valor_critico = st.chi2.ppf(1 - alpha, df)

print(f'Grados de libertad: {df}')
print(f'Valor crítico χ²(α=0.05, df={df}) = {valor_critico:.2f}')
print()
if chi2 < valor_critico:
    print(f'χ² = {chi2:.2f} < {valor_critico:.2f} → No se rechaza H₀.')
    print('El modelo sigmoide es consistente con los datos.')
else:
    print(f'χ² = {chi2:.2f} > {valor_critico:.2f} → Se rechaza H₀.')

El p-valor se aleja de los extremos 0 y 1, y el $\chi^2$ obtenido es menor que el valor crítico para $\alpha = 0.05$. Esto indica que el modelo sigmoide describe bien los datos: no hay evidencia para rechazar la hipótesis de que el modelo es correcto.

**9. Gráfico del ajuste con banda de error**

In [ ]:
fitter.plot_fit(
    xlabel='Longitud de onda $\\lambda$ (nm)',
    ylabel='Transmisión'
)
plt.title('Curva de transmisión del filtro óptico — ajuste sigmoide')
plt.show()

La banda de error es más ancha en la zona de transición (alrededor de $\lambda = a$), donde la pendiente de la sigmoide es máxima y la incerteza en los parámetros $a$ y $b$ tiene mayor impacto sobre el modelo.

---
## Conclusiones

Ajustamos la curva de transmisión de un filtro óptico pasa-altos con un modelo sigmoide mediante máxima verosimilitud binomial. Los parámetros obtenidos son:

- **Longitud de onda de corte:** $a = (\hat{a} \pm \delta a)\,\mathrm{nm}$
- **Pendiente de transición:** $b = (\hat{b} \pm \delta b)\,\mathrm{nm}$

El test $\chi^2$ confirma que el modelo es consistente con los datos. La banda de error del ajuste es mayor en la región de transición, donde la incerteza en los parámetros se traduce más fuertemente en incerteza sobre la transmisión predicha.